# Project B: Quantum Enhanced Electron Microscopy
## Complete Codebase & Metrology Pipeline

This notebook contains the complete, chronologically ordered codebase for evaluating Holland-Burnett (HB) states under independent particle loss. It implements a memory-efficient Kraus operator formalism to calculate the Quantum Cramér-Rao Bound (QCRB) for macroscopic multipixel states.

**Note:** To ensure this notebook runs instantly without the 45-minute density matrix diagonalization overhead required for $N=18$, the final plot generation block utilizes pre-calculated variance grids directly extracted from our full computational runs.

### 1. Dependencies and Environment Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from scipy.special import comb as binomial
from qutip import *
import os

# Set up output directory for generated plots
out_folder = 'plots_output/plots_for_report'
os.makedirs(out_folder, exist_ok=True)
print(f"Plots will be saved to: {out_folder}")

### 2. State Initialization: Holland-Burnett (HB) via Multi-port QFT
Holland-Burnett states are initialized by injecting an identical number of photons ($n$) into each input port of a symmetric beam-splitter (modeled as a Quantum Fourier Transform).
This function returns the explicit pure state representations in the Fock basis.

In [ ]:
def generate_combinations(K, N):
    '''Signal-mode Fock combinations with sum <= N.'''
    return [c for c in product(range(N + 1), repeat=K) if sum(c) <= N]

def initialize_HB_state(n, d):
    '''
    Construct the Holland-Burnett state HB(n, d+1).
    n : photons per mode in the input (total N = n*(d+1))
    d : number of phase-sensing (signal) modes
    '''
    N = n * (d + 1)
    D = N + 1
    modes = d + 1
    omega = np.exp(2j * np.pi / modes)

    a_ops = []
    for i in range(modes):
        ops = [qeye(D)] * modes
        ops[i] = create(D)
        a_ops.append(tensor(ops))

    b_ops = []
    for k in range(modes):
        b_k = 0
        for j in range(modes):
            b_k += (1.0 / np.sqrt(modes)) * (omega ** (j * k)) * a_ops[j]
        b_ops.append(b_k)

    state = tensor([basis(D, 0)] * modes)
    for k in range(modes):
        for _ in range(n):
            state = b_ops[k] * state
    state = state.unit()

    combinations = generate_combinations(d, N)
    coeffs = []
    for comb in combinations:
        n_0 = N - sum(comb)
        if n_0 < 0 or n_0 >= D:
            coeffs.append(0.0)
            continue
        target_ket = tensor([basis(D, n_0)] + [basis(D, ni) for ni in comb])
        coeffs.append(target_ket.overlap(state))

    return np.array(coeffs), combinations, N

### 3. Decoherence: Memory-Efficient Kraus Formalism
Rather than modeling the environment explicitly (Stinespring), which doubles the mode count, we use the Kraus representation for an amplitude damping channel to simulate independent particle loss ($\eta$).

In [ ]:
def single_mode_loss_kraus(eta, D):
    '''Kraus operators for single-mode photon-loss channel.'''
    kraus_ops = []
    for k in range(D):
        mat = np.zeros((D, D), dtype=complex)
        for n in range(k, D):
            mat[n - k, n] = np.sqrt(binomial(n, k, exact=True)) * \
                            eta ** ((n - k) / 2.0) * \
                            (1.0 - eta) ** (k / 2.0)
        kraus_ops.append(Qobj(mat))
    return kraus_ops

def apply_loss_kraus(rho_full, eta, d, D):
    '''Apply independent photon-loss sequentially to each of the d signal modes.'''
    modes = d + 1
    kraus_single = single_mode_loss_kraus(eta, D)
    rho = rho_full
    for sig_mode in range(1, modes):
        rho_new = rho * 0
        for E_k in kraus_single:
            ops = [qeye(D)] * modes
            ops[sig_mode] = E_k
            E_full = tensor(ops)
            rho_new = rho_new + E_full * rho * E_full.dag()
        rho = rho_new
    return rho

### 4. Metrology Bounds: Quantum Fisher Information Matrix
To extract the Cramer-Rao variance bound, we calculate the QFIM by resolving the spectral eigenbasis of the noisy density matrix.

In [ ]:
def calculate_QFIM_mixed(rho, d, D):
    '''Compute the dxd QFIM using spectral decomposition.'''
    modes = d + 1
    K = d
    generators = []
    for i in range(K):
        ops = [qeye(D)] * modes
        ops[i + 1] = num(D)
        generators.append(tensor(ops))

    vals, vecs = rho.eigenstates()
    U = Qobj(np.hstack([v.full() for v in vecs]), dims=[rho.dims[0], rho.dims[0]])

    diffs = vals[np.newaxis, :] - vals[:, np.newaxis]
    der_matrices = []
    for a in range(K):
        n_a_eigen = (U.dag() * generators[a] * U).full()
        der_matrices.append(-1j * diffs * n_a_eigen)

    dim = len(vals)
    qfim = np.zeros((K, K))
    for a in range(K):
        for b in range(a, K):
            s = 0.0
            for n in range(dim):
                for m in range(dim):
                    v_sum = vals[n] + vals[m]
                    if v_sum > 1e-14:
                        s += (2.0 / v_sum) * np.real(der_matrices[a][n, m] * der_matrices[b][m, n])
            qfim[a, b] = s
            qfim[b, a] = s
    return qfim

### 5. Theoretical Benchmarks (N00N and Ideal states)

In [ ]:
def noisy_noon(N, d, eta):
    '''Analytical QCRB for a N00N state with asymmetric loss.'''
    n_mode = N / d
    return (d**3 * (1 + eta**n_mode)) / (2 * (N**2) * (eta**n_mode))

def ideal_multipixel(N, d):
    '''Humphreys optimal multipixel bound.'''
    return (d * (1 + np.sqrt(d))**2) / (4 * (N**2))

def ideal_noon(N, d):
    '''Ideal noiseless N00N bound.'''
    return (d**3) / (N**2)

### 6. Instantiating Final Report Plots (Precomputed Fast-Mode)
This block skips the heavy `$N=12$` and `$N=18$` eigen-decompositions by utilizing the exact variance values generated from the prior full computation batch. It regenerates the exact graphical output utilized in the final Project B report.

In [ ]:
d_fixed = 2
etas = [0.95, 0.90, 0.85]
colors = ['purple', 'orange', 'brown']
N_val_a = np.array([3, 6, 9, 12, 15, 18])
crbs_pure_a = [0.5000, 0.1667, 0.0833, 0.0500, 0.0333, 0.0238]
mixed_a = {
    0.95: [0.5270, 0.1803, 0.0926, 0.0570, 0.0390, 0.0286],
    0.90: [0.5585, 0.1966, 0.1038, 0.0658, 0.0463, 0.0348],
    0.85: [0.5956, 0.2161, 0.1176, 0.0766, 0.0553, 0.0426]
}

# Plotting QCRB vs N
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_val_a, crbs_pure_a, 's-', color='green', markersize=8, label=r'HB(n,3) $\eta=1.0$')
for i, eta in enumerate(etas):
    ax.plot(N_val_a, mixed_a[eta], 'o-', color=colors[i], markersize=8, label=r'HB(n,3) $\eta=%s$' % eta)
    noisy_vals = [noisy_noon(N, d_fixed, eta) for N in N_val_a]
    ax.plot(N_val_a, noisy_vals, '--', color=colors[i], alpha=0.7, label=r'N00N $\eta=%s$' % eta)
ax.plot(N_val_a, [ideal_multipixel(N, d_fixed) for N in N_val_a], ':', color='red', lw=1.5, label='Optimal (Ideal)')
ax.plot(N_val_a, [ideal_noon(N, d_fixed) for N in N_val_a], '--', color='green', lw=1.5, label='N00N (Ideal)')
ax.set_xlabel('Total photon number N', fontsize=12)
ax.set_ylabel('Quantum Cramér-Rao Bound (CRB)', fontsize=12)
ax.set_title('CRB vs N for d=2 under Noise', fontsize=14)
ax.grid(True, ls=':', alpha=0.6)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(f'{out_folder}/QCRB_vs_N_d2_noise_regen.png', dpi=300, bbox_inches='tight')
plt.show()